# 实验七 · 矩阵向量乘法 —— 嵌套循环并行与 SIMD 协同

**所属**：《并行计算技术》第五章 · OpenMP 编程　|　**难度**：⭐⭐⭐ 重点　|　**预计时长**：30–40 分钟

前六个实验主要讨论**线程级并行**，即如何将工作分配到多个核心。每个核心内部还存在向量部件，第三章的 ARM NEON 即属于这一层次。本实验说明线程级并行与数据级并行彼此**正交**、可以叠加使用，并给出在嵌套循环中合理安排二者的方式。此外，本实验还将分析本章第一个**访存受限**的负载，其加速比上限由内存带宽而非核心数决定。

本实验只有一个计算内核 $y = Ax$，但有六个版本、三组运行参数。贯穿全篇的结论是：**最优的并行方案取决于矩阵的形状，而不只取决于算法**。同一份代码，在方阵上的最佳选择与在扁矩阵上的最佳选择并不相同。

> **实验说明**
> 1. 本实验采用**递进式的版本组织**：以串行实现为基准，每个版本仅引入一种新的 OpenMP 构造或一处相应的代码改写，并在同一次运行中完成全部版本的计时与正确性校验，因而各版本面对的是完全相同的数据与运行环境，各版本之间具有可比性。
> 2. 请自上而下依次执行各单元格（Shift+Enter）。
> 3. 本实验依赖支持 OpenMP 的 **GCC 编译器**，建议在华为鲲鹏处理器或其他 AArch64 平台上运行。
> 4. 遇到 🔧 **动手练习** 与 🤔 **思考题** 时，建议先独立完成，再阅读后续内容。
> 5. 本实验只有**一个**可执行程序 `omp_gemv`，六个版本全部内置其中。三次运行改变的只是命令行参数 `<m> <n> <thread_count>`。
> 6. 矩阵占用内存约为 $8mn$ 字节(`double`)，其中最大的一组为 4000×4000（约 122 MB）。可按目标平台的内存容量调整 8.2 节中的 `M_SQ`、`N_SQ`。
> 7. 默认线程数按 **16 核**平台设定（`NT = 16`）。若平台核心数不同，请修改该变量；「扁矩阵」一组要求行数 $m$ **小于**线程数，才能体现 `collapse` 的价值。


## 🎯 学习目标

完成本实验后，学生应能够：

- 说明矩阵向量乘法的三种划分方式（按行、按列、二维），并比较它们在可用线程数、归约需求与并行区域数上的差异
- 定量分析「并行内层」的代价：$m$ 次并行区域、$m$ 次归约、$m$ 次隐式栅栏，并说明该代价为何与 $m$ 成正比、与 $n$ 无关
- 掌握 `#pragma omp simd` 的语义：它是对编译器的**授权**，而非请求
- 解释为何 `-O3` 在没有 `-ffast-math` 时不会自动向量化浮点归约循环，以及 `simd reduction` 子句如何改变这一状况
- 理解**线程级并行**（TLP）与**数据级并行**（DLP）的正交关系，并能写出二者叠加的标准形式
- 掌握 `collapse(k)` 子句的语义与适用条件，说明它为何不能直接用于标准 GEMV，以及如何通过二维划分把内层归约改造为独立的迭代空间
- 用 Roofline 模型判断一个内核是计算受限还是访存受限，并据此预估其加速比上限
- 说明**最优并行方案取决于问题形状**这一结论，并能为给定形状的矩阵选择合适的版本


## 🗺️ 学习路径

1. **背景知识**：GEMV 两层循环的性质、矩阵形状与可用并行度、Roofline 分析
2. **关键知识点**：并行层次的选择、`collapse(k)`、`#pragma omp simd`、TLP 与 DLP 的正交性
3. **六个版本**：V0 串行 → V1 按行划分 → V2 按列划分 → V3 按行 + SIMD → V4 二维 + `collapse(2)` → V5 二维 + `collapse` + SIMD
4. **运行一 · 方阵**（4000×4000，16 线程）：$m \gg p$，观察 V2 的崩塌与 SIMD 的净收益
5. **运行二 · 扁矩阵**（4×1000000，16 线程）：$m < p$，观察 `collapse` 的价值，以及 V2 为何在此不再是反模式
6. **对照 · 中等形状**（64×62500，16 线程）：$m > p$，`collapse` 的收益消失
7. **结果分析**：三组结果合起来给出「按矩阵形状选择方案」的判据


## 1. 背景知识：矩阵向量乘法

### 1.1 算法

$$y_i = \sum_{j=0}^{n-1} A_{ij} \cdot x_j, \qquad i = 0, 1, \ldots, m-1$$

```c
for (long i = 0; i < m; i++) {
  double sum = 0.0;
  const double *row = A + i * n;
  for (long j = 0; j < n; j++) {
    sum += row[j] * x[j];
  }
  y[i] = sum;
}
```

外层循环遍历行，内层循环完成一次向量内积。

### 1.2 两层循环的性质对比

| | 外层（`i`） | 内层（`j`） |
|---|---|---|
| 迭代次数 | $m$ | $n$ |
| 迭代之间是否独立 | **是**，各行互不影响 | **否**，共同累加到 `sum` |
| 并行化所需 | 无需归约 | 必须归约 |
| 写入目标 | `y[i]`，各线程互不重叠 | 同一个 `sum` |

由上表可见：外层循环的各次迭代相互独立，适合作为并行化的优先选择；内层循环虽然也可并行，但需要归约，且并行区域将被重复创建 $m$ 次。

### 1.3 矩阵形状决定可用的并行度

上面的结论有一个隐含前提：**外层的迭代次数 $m$ 足够多**。若 $m$ 小于线程数 $p$，按行划分就用不满线程组。设想 $m = 4$、$p = 16$：十二个线程无事可做，无论内层有多少工作。

因此需要在算法层面考虑三种划分方式：

```text
      按行划分                 按列划分                  二维划分
  ┌───────────────┐       ┌───┬───┬───┬───┐       ┌───┬───┬───┬───┐
  │ 线程 0        │       │ 0 │ 1 │ 2 │ 3 │       │ 0 │ 1 │ 2 │ 3 │
  ├───────────────┤       │   │   │   │   │       ├───┼───┼───┼───┤
  │ 线程 1        │       │   │   │   │   │       │ 4 │ 5 │ 6 │ 7 │
  ├───────────────┤       │   │   │   │   │       ├───┼───┼───┼───┤
  │ 线程 2        │       │   │   │   │   │       │ 8 │ 9 │10 │11 │
  └───────────────┘       └───┴───┴───┴───┘       └───┴───┴───┴───┘
   每个线程整行            每个线程一段列          每个线程一个块
```

| | 按行划分 | 按列划分 | 二维划分 |
|---|---|---|---|
| 可用线程数 | $\min(m, p)$ | $p$ | $p$ |
| 是否需要归约 | 否 | **是**，每行一次 | 需一次廉价的部分和合并 |
| 并行区域数 | 1 | **$m$** | 1 |
| 适用形状 | $m \gg p$ | $m$ 很小 | 任意，尤其 $m < p$ |

按行划分与二维划分（棋盘划分）是矩阵向量乘法的两种经典并行方案。本实验把三种方式都实现出来，并在不同形状的矩阵上实测其表现。

### 1.4 Roofline 分析：这是一个访存受限的内核

回顾第一章的 Roofline 模型。计算内核的**算术强度**定义为：

$$I = \frac{\text{浮点运算次数}}{\text{与主存之间传输的字节数}}$$

> 需要注意分母的确切含义：它是穿过各级缓存、真正到达**主存**的流量，而不是程序发出的访存请求总量。本内核中 $A$ 的每个元素只被使用一次，缓存无法提供任何复用，二者恰好相等；而在具有数据复用的内核中（例如分块后的矩阵乘法），主存流量会远小于访存总量，算术强度也随之提高。

对本内核而言：

| 项 | 数量 |
|---|---|
| 浮点运算 | $2mn$（每次乘加计 2 次） |
| 主存流量 | $8mn$ 字节（矩阵 $A$，双精度，且每个元素只用一次） |
| 算术强度 | $I = 2mn / 8mn = 0.25$ flop/byte |

> 向量 $x$ 与结果 $y$ 的访存量相对于 $A$ 可以忽略：$x$ 长度为 $n$ 且可常驻缓存，$y$ 长度为 $m$ 且只写一次。

$0.25$ flop/byte 是一个很低的算术强度。以典型嵌入式 ARM 平台的内存带宽 10 GB/s 计，该内核的性能上限约为 $10 \times 0.25 = 2.5$ GFLOPS，远低于处理器的峰值浮点能力。

**由此可得出以下两点推论**：

1. **多线程的加速比会显著低于线程数**。增加核心数并不会增加内存带宽；当带宽已经饱和时，额外的核心主要处于等待数据的状态。
2. **向量化的收益同样有限**。SIMD 主要加速计算部分，而本内核的瓶颈并不在计算。

第四章的 GEMV 实验已经遇到过同一现象，本实验将在 OpenMP 的语境下再次验证它。这一点在第 9 节还会再次出现：`collapse` 能提高线程利用率，却提不高带宽。


## 2. 嵌套循环中的并行层次

### 2.1 并行外层还是并行内层

```c
// 方案一：并行外层  —— 推荐
#pragma omp parallel for
for (i = 0; i < m; i++)
  for (j = 0; j < n; j++) { ... }

// 方案二：并行内层  —— 反模式
for (i = 0; i < m; i++)
#pragma omp parallel for reduction(+ : sum)
  for (j = 0; j < n; j++) { ... }
```

| | 方案一 | 方案二 |
|---|---|---|
| 并行区域创建次数 | 1 | **$m$** |
| 隐式栅栏次数 | 1 | **$m$** |
| 归约次数 | 0 | **$m$** |
| 单个线程连续处理的数据 | 整行，缓存友好 | 行的片段 |

以 $m = 4000$ 计，方案二将创建约四千次并行区域。结合实验五在本平台测得的单次 Fork-Join 代价（约 $8\text{–}10\,\mu s$），仅此一项开销即达数十毫秒，而整个串行计算的耗时通常仅十余毫秒。该代价与平台及线程数相关，建议以自己在实验五中的实测值重新估算。

**一般原则**：在嵌套循环中，把并行放在**迭代次数足够多、且迭代之间独立**的最外层循环上。

请注意这条原则里的限定语「迭代次数足够多」。方案二的代价与外层迭代次数 $m$ 成正比，而与内层规模 $n$ 无关；因此当 $m$ 很小时，这项代价也随之变小，方案二未必仍是反模式。第 9 节的运行二将实测这一点。

### 2.2 `collapse(k)` 子句

当外层循环的迭代次数**少于线程数**时，把并行放在外层就无法用满全部线程。`collapse(k)` 把最外的 $k$ 层循环合并为一个迭代空间：

```c
#pragma omp parallel for collapse(2)
for (long i = 0; i < m; i++)        // m = 4
  for (long j = 0; j < n; j++)      // n = 500000
    c[i * n + j] = sin(a[i * n + j]);
// 合并后的迭代空间为 m × n = 2000000 个点
```

**使用条件**（必须全部满足）：

1. **完美嵌套**：两层循环头之间不得有任何语句；
2. **循环界的限制**：在 OpenMP 4.5 及更早的规范中，内层循环的上下界不得依赖外层循环变量，因此三角形循环 `for (j = i; j < n; j++)` 不能 collapse。OpenMP 5.0 起引入了**非矩形循环嵌套**（non-rectangular loop nest）支持，允许内层循环界为外层循环变量的线性表达式（GCC 自 11 起支持）。使用前应先确认目标编译器所遵循的规范版本，可用 `_OPENMP` 宏判断；
3. 每一层都必须满足实验二 3.6 节所述的`parallel for`规范形式。

**何时使用**：外层迭代次数少、内层迭代次数多，且两层都独立。若外层迭代次数已经远大于线程数，`collapse` 不会带来收益，反而可能因索引计算开销增加而略有下降。

> **标准写法的矩阵向量乘法不能直接使用 `collapse(2)`**，原因有两重：两个循环头之间有局部变量声明，不满足完美嵌套；且内层迭代共同累加到 `sum`，并不独立。第二点更为根本——**`collapse` 只负责合并迭代空间，它不会替你消除归约**。
>
> 但这并不意味着 GEMV 与 `collapse` 无缘。把每行的内积拆成若干列块、各块的部分和写入独立位置之后，`(行, 列块)` 这两层就同时满足了上述全部条件。7.5 节的 V4 正是这样做的。

## 3. 数据级并行与 TLP/DLP 协同

### 3.1 `#pragma omp simd`

```c
#pragma omp simd reduction(+ : sum)
for (long j = 0; j < n; j++) {
  sum += row[j] * x[j];
}
```

`simd` 构造**允许**编译器把该循环转换为 SIMD 循环，即把连续的多次迭代合并为一条向量指令执行。它与 `for` 构造的分工是：

| 构造 | 分配对象 | 并行层次 |
|---|---|---|
| `#pragma omp for` | 迭代 → 线程 | 线程级（TLP） |
| `#pragma omp simd` | 迭代 → 向量通道 | 数据级（DLP） |

**关键点在于：`simd` 是对编译器的授权，而非请求。**使用该制导语句意味着程序员向编译器声明：该循环的迭代之间不存在阻碍向量化的依赖，并且**允许改变浮点运算的结合次序**。是否真的实施向量化，仍由编译器根据向量宽度、访存模式与成本模型决定。

### 3.2 为何 `-O3` 不会自动向量化这个循环

这一点对于理解本实验尤为重要。

内层循环 `sum += row[j] * x[j]` 是一个**浮点归约**。要把它向量化，必须把 $n$ 次串行的加法改组为「若干个向量通道各自累加，最后横向合并」。这一改组**改变了浮点加法的次序**。

浮点加法不满足结合律：

$$(a + b) + c \ne a + (b + c) \quad \text{（在有限精度下）}$$

因此，在未获得明确授权的情况下，编译器**不会**进行这一改组，以保证结果与串行执行完全一致。

解除这一限制有两种途径：

| 途径 | 作用范围 | 风险 |
|---|---|---|
| `-ffast-math` | **全局**，整个编译单元 | 影响所有浮点运算，可能破坏数值稳定性 |
| `#pragma omp simd reduction(+ : sum)` | **仅此循环** | 可控，风险局限在该循环 |

本实验统一使用 `-O3 -fopenmp`，**不加** `-ffast-math`。因此 V1 的内层循环不会被向量化，而 V3 的会。**两者的耗时之差可归因于向量化本身。**

> 这一设计把「是否允许运算重排」作为单一变量加以隔离，便于对照分析。

### 3.3 TLP 与 DLP 的正交性

```c
#pragma omp parallel for              // 第一层：行 → 线程
for (long i = 0; i < m; i++) {
  double sum = 0.0;
#pragma omp simd reduction(+ : sum)   // 第二层：列 → 向量通道
  for (long j = 0; j < n; j++) {
    sum += row[j] * x[j];
  }
  y[i] = sum;
}
```

```text
          ┌──────── 核心 0 ────────┐  ┌──────── 核心 1 ────────┐
   TLP →  │  行 0, 1, ..., m/2-1   │  │  行 m/2, ..., m-1      │
          │  ┌──┬──┬──┬──┐         │  │  ┌──┬──┬──┬──┐         │
   DLP →  │  │d0│d1│d2│d3│ 向量通道│  │  │d0│d1│d2│d3│         │
          │  └──┴──┴──┴──┘         │  │  └──┴──┴──┴──┘         │
          └────────────────────────┘  └────────────────────────┘
```

两层并行相乘：$p$ 个核心 × $w$ 个向量通道 = 理论上 $p \times w$ 倍。AArch64 的 NEON 寄存器宽 128 位，可容纳 2 个双精度数，因此 $w = 2$。

> **实际收益远低于该理论值**，原因在于本内核受带宽限制，这与 1.3 节 Roofline 分析的结论一致。


## 4. 环境准备与检查

本节确认三项内容：编译器是否支持 OpenMP、运行时报告的处理器数量、以及各处理器核心的最大频率是否一致。

第三项检查针对**异构多核**平台。Arm 的 big.LITTLE 架构把高性能核心与高能效核心集成在同一块芯片上，二者的频率与微架构均不相同。在这类平台上，同一段代码在不同类型的核心上执行，耗时可能相差 2 倍以上，线程数与加速比之间因而不再是简单的线性关系。

需要说明的是，最大频率不一致只是异构多核的**必要非充分**证据：同构多核平台也可能因加速频率（boost）策略或芯片分级（binning）而上报不同的 `cpuinfo_max_freq`。因此下面的检查只给出提示，确认平台是否为异构架构还需结合 `lscpu` 输出的核心型号信息。

In [ ]:
import os
import re
import subprocess
import platform

print('=' * 60)
print(' 一、平台信息')
print('=' * 60)
print('操作系统   :', platform.system(), platform.release())
print('处理器架构 :', platform.machine())
print('逻辑核心数 :', os.cpu_count())

print()
print('=' * 60)
print(' 二、编译器与 OpenMP 支持')
print('=' * 60)
gcc_ver = subprocess.run(['gcc', '--version'], capture_output=True,
                         text=True).stdout.splitlines()[0]
print('编译器     :', gcc_ver)

probe = subprocess.run('echo | gcc -fopenmp -dM -E -x c - | grep _OPENMP',
                       shell=True, capture_output=True, text=True).stdout.strip()
if probe:
    ver = int(probe.split()[-1])
    spec = {200805: '3.0', 201107: '3.1', 201307: '4.0',
            201511: '4.5', 201811: '5.0', 202011: '5.1'}.get(ver, '未知')
    print('_OPENMP    :', ver, '(对应 OpenMP %s 规范)' % spec)
    print('数组段归约 :', '支持' if ver >= 201511 else '不支持（需要 4.5 及以上）')
else:
    print('⚠️  未检测到 OpenMP 支持，请确认编译时带有 -fopenmp')

print()
print('=' * 60)
print(' 三、核心频率与异构性检查')
print('=' * 60)
freqs = []
for cpu in range(os.cpu_count() or 1):
    path = '/sys/devices/system/cpu/cpu%d/cpufreq/cpuinfo_max_freq' % cpu
    try:
        with open(path) as f:
            freqs.append((cpu, int(f.read().strip()) // 1000))
    except OSError:
        pass

if not freqs:
    print('无法读取 cpufreq 节点，跳过异构性检查。')
else:
    for cpu, mhz in freqs:
        print('  CPU%-2d 最大频率: %5d MHz' % (cpu, mhz))
    distinct = sorted(set(m for _, m in freqs))
    if len(distinct) > 1:
        print()
        print('⚠️  检测到 %d 种不同的最大频率，本平台可能为异构多核架构（如 Arm big.LITTLE）。'
              % len(distinct))
        print('    请结合 lscpu 输出的核心型号信息进一步确认。')
        print('    若确为异构平台，测速前建议执行：')
        print('      export OMP_PROC_BIND=close')
        print('      export OMP_PLACES=cores')
    else:
        print()
        print('✅ 全部核心的最大频率一致，可按同构多核平台处理。')

print()
print('OMP_NUM_THREADS =', os.environ.get('OMP_NUM_THREADS', '（未设置，由运行时决定）'))
print('OMP_PROC_BIND   =', os.environ.get('OMP_PROC_BIND', '（未设置）'))
print('OMP_PLACES      =', os.environ.get('OMP_PLACES', '（未设置）'))

## 5. 实验工具函数

本节定义三个贯穿全章的辅助函数，后续各实验均直接调用，不再重复说明。

| 函数 | 作用 |
|---|---|
| `compile_c(src)` | 以 `-O3 -fopenmp -Wall -Wextra` 编译指定源文件，并回显全部告警 |
| `run_c(binary, *args)` | 运行可执行文件并原样打印其标准输出 |
| `parse_table(output)` | 从程序输出的结果表中提取「方法名 / 耗时 / 加速比 / 校验」四列 |

**关于编译选项**：全章统一使用 `-O3 -fopenmp`。AArch64 平台的 NEON 属于基线指令集，无需附加 `-march` 或 `-mcpu` 选项。`-Wall -Wextra` 用于暴露数据环境声明不当引发的告警，这类告警在 OpenMP 程序中往往是并发缺陷的征兆，不应忽略。

In [ ]:
import subprocess
import re
import os

SRC_DIR = 'src_gemv'
os.makedirs(SRC_DIR, exist_ok=True)

CFLAGS = ['-O3', '-fopenmp', '-Wall', '-Wextra']


def compile_c(src, extra=('-lm',)):
    """编译单个源文件，返回可执行文件路径；编译失败时抛出异常。"""
    src_path = os.path.join(SRC_DIR, src)
    binary = os.path.join(SRC_DIR, os.path.splitext(src)[0])
    cmd = ['gcc'] + CFLAGS + ['-o', binary, src_path] + list(extra)
    print('$', ' '.join(cmd))
    proc = subprocess.run(cmd, capture_output=True, text=True)
    if proc.stdout.strip():
        print(proc.stdout.rstrip())
    if proc.stderr.strip():
        print(proc.stderr.rstrip())
    if proc.returncode != 0:
        raise RuntimeError('编译失败：%s' % src)
    print('✅ 编译通过，无告警' if not proc.stderr.strip()
          else '⚠️  编译通过，但存在告警，请逐条阅读')
    return binary


def run_c(binary, *args, env=None):
    """运行可执行文件，打印并返回其标准输出。"""
    cmd = [binary] + [str(a) for a in args]
    print('$', ' '.join(cmd))
    print()
    run_env = dict(os.environ)
    if env:
        run_env.update({k: str(v) for k, v in env.items()})
    proc = subprocess.run(cmd, capture_output=True, text=True, env=run_env)
    print(proc.stdout.rstrip())
    if proc.stderr.strip():
        print('[stderr]', proc.stderr.rstrip())
    return proc.stdout


ROW_RE = re.compile(r'^\|\s*(.+?)\s*\|\s*([0-9.]+)\s*\|\s*([0-9.]+)x\s*\|\s*(\S+)\s*\|$')


def parse_table(output):
    """解析结果表，返回 [(方法名, 耗时ms, 加速比, 校验结论), ...]。"""
    rows = []
    for line in output.splitlines():
        m = ROW_RE.match(line.strip())
        if m:
            rows.append((m.group(1), float(m.group(2)),
                         float(m.group(3)), m.group(4)))
    return rows


print('工具函数已就绪，源码目录：', os.path.abspath(SRC_DIR))

In [ ]:
import matplotlib
import matplotlib.pyplot as plt

matplotlib.rcParams['font.sans-serif'] = ['DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False

C_BASE, C_GOOD, C_FAIL, C_SLOW = '#7f7f7f', '#1f77b4', '#d62728', '#ff7f0e'


def plot_speedup(rows, title, figsize=(10, 5)):
    """绘制加速比柱状图。配色：灰=基准，蓝=有效加速，橙=慢于基准，红=校验失败。"""
    if not rows:
        print('未解析到结果行，请先运行上一单元格。')
        return
    names = [r[0] for r in rows]
    speeds = [r[2] for r in rows]
    colors = []
    for i, (_, _, sp, chk) in enumerate(rows):
        if i == 0:
            colors.append(C_BASE)
        elif chk == 'FAIL':
            colors.append(C_FAIL)
        elif sp < 1.0:
            colors.append(C_SLOW)
        else:
            colors.append(C_GOOD)

    fig, ax = plt.subplots(figsize=figsize)
    bars = ax.bar(range(len(names)), speeds, color=colors,
                  edgecolor='black', linewidth=0.6, width=0.6)
    ax.axhline(1.0, color='black', linestyle='--', linewidth=1.0, alpha=0.7)
    ax.set_xticks(range(len(names)))
    ax.set_xticklabels(names, rotation=20, ha='right', fontsize=9)
    ax.set_ylabel('Speedup vs. Serial Baseline')
    ax.set_title(title, fontsize=12, pad=12)
    ax.grid(axis='y', linestyle=':', alpha=0.5)
    ax.set_axisbelow(True)

    for bar, (_, ms, sp, chk) in zip(bars, rows):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() * 1.02,
                '%.2fx\n%.1f ms%s' % (sp, ms, '' if chk in ('-', 'PASS') else '\nFAIL'),
                ha='center', va='bottom', fontsize=8)

    ax.set_ylim(0, max(speeds) * 1.30)
    plt.tight_layout()
    plt.show()


print('绘图函数已就绪。配色：灰=基准，蓝=有效加速，橙=慢于基准，红=校验失败。')

## 6. 版本设计总览

本实验只有一个计算内核，六个版本对应六种并行组织方式。它们共用同一份数据、同一次进程、同一套计时与校验代码。

| 版本 | 划分方式 | 并行构造 | 可用线程数 | 并行区域数 | 向量化 | 结果表行号 |
|---|---|---|---|---|---|---|
| **V0** | 串行 | — | 1 | 0 | 否 | 1 |
| **V1** | 按行 | `parallel for`（外层 `i`） | $\min(m, p)$ | 1 | 否 | 2 |
| **V2** | 按列 | `parallel for reduction`（内层 `j`） | $p$ | **$m$** | 否 | 3 |
| **V3** | 按行 | `parallel for` + `simd` | $\min(m, p)$ | 1 | **是** | 4 |
| **V4** | 二维 | `parallel for collapse(2)` | $p$ | 1 | 否 | 5 |
| **V5** | 二维 | `parallel for collapse(2)` + `simd` | $p$ | 1 | **是** | 6 |

**五组对照的读法**：

| 对照 | 度量的对象 |
|---|---|
| V0 → V1 | 线程级并行的净收益（受带宽限制） |
| V1 → V2 | 划分方式的代价对比：按列划分的开销与 $m$ 成正比 |
| V1 → V3 | **向量化的净收益**（按行划分下，其余条件完全相同） |
| V4 → V5 | **向量化的净收益**（二维划分下），用于验证 TLP 与 DLP 的正交性 |
| V1 → V4 | 二维划分 + `collapse(2)` 的净收益 |

最后一组对照需要两次运行才能读懂。在**方阵**上（$m \gg p$）按行划分本已用满线程，此时 V4 与 V1 应当接近——这说明**二维划分这一代码重构本身是性能中性的**。有了这个基准，**扁矩阵**上 V4 相对 V1 的差距，才能干净地归因于 `collapse(2)` 带来的线程利用率提升，而不是归因于重构。

**三组运行参数**：

| 运行 | 形状 | $m$ | $n$ | 线程数 | $m$ 与 $p$ 的关系 | 观察重点 |
|---|---|---|---|---|---|---|
| 一 | 方阵 | 4000 | 4000 | 16 | $m \gg p$ | V2 的崩塌、SIMD 的净收益、V4 ≈ V1 |
| 二 | 扁矩阵 | 4 | 1000000 | 16 | $m < p$ | `collapse` 的价值、V2 不再是反模式 |
| 三 | 中等 | 64 | 62500 | 16 | $m > p$ | `collapse` 的收益消失 |

运行二与运行三的矩阵元素总数相同（均为 $4 \times 10^6$），因此计算量相同，唯一变化的是形状。


## 7. 逐版本代码讲解

### 7.1 V0 · 串行基准

```c
for (long i = 0; i < m; i++) {
  double sum = 0.0;
  const double *row = A + i * n;
  for (long j = 0; j < n; j++) {
    sum += row[j] * x[j];
  }
  y[i] = sum;
}
```

`const double *row = A + i * n;` 把行首指针提出来，使内层循环的访存形式变为一维顺序访问，便于编译器优化。

### 7.2 V1 · 按行划分

```c
#pragma omp parallel for num_threads(thread_count) default(none) \
    shared(A, x, y, m, n)
for (long i = 0; i < m; i++) {
  double sum = 0.0;              // 声明于循环体内，自然具有私有属性
  const double *row = A + i * n;
  for (long j = 0; j < n; j++) {
    sum += row[j] * x[j];
  }
  y[i] = sum;                    // 各线程写不同的 i，互不重叠
}
```

**以下三点需要注意**：

1. **不需要 `reduction`**。`sum` 是每一行自己的局部累加器，不同的行之间没有共享。这正是并行外层的优势。
2. **`sum` 与 `row` 声明在循环体内**，按实验二的默认规则自动私有，无需任何子句。
3. **`y[i]` 的写入不存在竞争**，因为静态划分保证每个 `i` 只属于一个线程。但要注意：若 $m$ 很小而线程数很多，相邻的 `y[i]` 可能落在同一条缓存行内，从而引发实验六讨论的伪共享。本实验中每个线程连续处理多行，且每行只写一次，因此影响可以忽略。

### 7.3 V2 · 按列划分

```c
for (long i = 0; i < m; i++) {
  double sum = 0.0;
  const double *row = A + i * n;

#pragma omp parallel for num_threads(thread_count) \
    reduction(+ : sum) default(none) shared(row, x, n)
  for (long j = 0; j < n; j++) {
    sum += row[j] * x[j];
  }
  y[i] = sum;
}
```

这一版本在语法上完全正确，校验也会通过。它把并行放在内层，因此：

- 并行区域被创建 $m$ 次；
- 归约执行 $m$ 次；
- 隐式栅栏经历 $m$ 次。

**关键在于这项代价的量纲**：它正比于 $m$，而与 $n$ **无关**。$m = 4000$ 时，仅 Fork-Join 一项就约合数十毫秒，程序被彻底拖垮；而 $m = 4$ 时，同样一项只有几十微秒，完全可以忽略。

因此，把 V2 简单地称作「反模式」并不准确。它与实验五 V1 是同一个结构（在高频循环内部创建并行区域），而实验五给出的判据是**并行区域的创建次数与每次区域内有效工作量之比**。在方阵上这个比值很糟，在扁矩阵上则相当合理。第 9 节的两次运行会把这一点实测出来。

### 7.4 V3 · 按行划分 + 内层向量化

```c
#pragma omp parallel for num_threads(thread_count) default(none) \
    shared(A, x, y, m, n)
for (long i = 0; i < m; i++) {
  double sum = 0.0;
  const double *row = A + i * n;

#pragma omp simd reduction(+ : sum)
  for (long j = 0; j < n; j++) {
    sum += row[j] * x[j];
  }
  y[i] = sum;
}
```

与 V1 相比只增加了一行 `#pragma omp simd`。这一行授权编译器改变浮点加法的次序，从而生成向量指令。

**注意 `reduction` 子句在此处的含义**：它不是线程之间的归约，而是**向量通道之间**的归约。编译器会为每个通道维护一个部分和，循环结束后横向相加。

> **在 AArch64 上无需附加编译选项**。NEON 属于 AArch64 的基线指令集，编译器默认即可生成 NEON 指令，不需要 `-march` 或 `-mcpu`。这与 x86 平台需要 `-mavx2` 之类的选项不同。

### 7.5 V4 · 二维划分 + `collapse(2)`

2.2 节已经指出，标准写法的 GEMV 不能直接 `collapse(2)`，原因有两重：

```c
for (long i = 0; i < m; i++) {
  double sum = 0.0;                  // ← 语句位于两个循环头之间
  const double *row = A + i * n;     // ← 同上
  for (long j = 0; j < n; j++) {
    sum += row[j] * x[j];            // ← 内层迭代共同累加到 sum
  }
  y[i] = sum;
}
```

- **不满足完美嵌套**：两个循环头之间有两条语句，`collapse(2)` 在语法上即不被接受；
- **内层迭代不独立**：即便把这两条语句挪走，内层的 $n$ 次迭代仍共同累加到同一个 `sum`。两层合并之后，各线程会并发更新同一个变量，语义随之出错。

第二点更为根本：**`collapse` 只负责合并迭代空间，它不会替你消除归约**。

解决办法是把每一行的内积拆成 $nb$ 个**列块**，各块的部分和写入 `partial` 数组，最后再把同一行的 $nb$ 个部分和相加：

```c
#pragma omp parallel for collapse(2) num_threads(thread_count) default(none) \
    shared(A, x, partial, m, n, nb)
for (long i = 0; i < m; i++) {
  for (long b = 0; b < nb; b++) {          // 两个循环头之间没有任何语句
    double s = 0.0;
    const double *row = A + i * n;
    const long lo = b * n / nb;            // 由 b 直接算出，不要求 n 被 nb 整除
    const long hi = (b + 1) * n / nb;
    for (long j = lo; j < hi; j++) {
      s += row[j] * x[j];
    }
    partial[i * nb + b] = s;               // 每个 (i, b) 写各自的位置
  }
}

for (long i = 0; i < m; i++) {             // 第二遍：合并部分和
  double sum = 0.0;
  for (long b = 0; b < nb; b++) sum += partial[i * nb + b];
  y[i] = sum;
}
```

改造之后，`(i, b)` 这两层满足 `collapse(2)` 的全部条件：

| 条件 | 是否满足 | 说明 |
|---|---|---|
| 完美嵌套 | ✅ | `i` 与 `b` 两个循环头之间没有任何语句，局部变量声明已移入 `b` 循环体内 |
| 循环界不变 | ✅ | `b` 的上下界为 `0` 与 `nb`，与 `i` 无关 |
| 迭代相互独立 | ✅ | 每个 $(i, b)$ 只写 `partial[i * nb + b]`，互不重叠 |

合并后的迭代空间有 $m \times nb$ 个点。程序取 $nb = $ 线程数，因此即便 $m = 1$ 也足以分给全部线程。

> **关于 `partial` 数组的伪共享**：相邻的 `partial[i * nb + b]` 由不同线程写入，落在同一条缓存行内，理论上存在实验六讨论的伪共享。但每个 $(i, b)$ 任务包含约 $n / nb$ 次乘加，而对 `partial` 只写一次——按实验六 3.1 节的判据，写入频率极低，其影响可以忽略。这是该判据的一次典型应用。

**第二遍合并的开销**：共 $m \times nb$ 次加法。以运行二的参数计为 $4 \times 16 = 64$ 次，相对于 $4 \times 10^6$ 次乘加完全可以忽略，因此保持串行即可。

### 7.6 V5 · 二维划分 + `collapse(2)` + SIMD

```c
#pragma omp parallel for collapse(2) num_threads(thread_count) default(none) \
    shared(A, x, partial, m, n, nb)
for (long i = 0; i < m; i++) {
  for (long b = 0; b < nb; b++) {
    double s = 0.0;
    const double *row = A + i * n;
    const long lo = b * n / nb;
    const long hi = (b + 1) * n / nb;

#pragma omp simd reduction(+ : s)          // ← 与 V4 相比只增加这一行
    for (long j = lo; j < hi; j++) {
      s += row[j] * x[j];
    }
    partial[i * nb + b] = s;
  }
}
```

`collapse(2)` 合并的是 `i` 与 `b` 两层，最内层的 `j` 循环位于循环体内部，`simd` 作用其上完全合法。

这一版的意义不只是「再快一点」。V1 → V3 与 V4 → V5 是同一件事的两次演示：**无论线程级并行以「按行划分的 `for`」还是「二维划分的 `collapse`」形式出现，都能与数据级并行叠加**。一次演示是个例，两次才说明二者确实正交。

### 7.7 关于校验容差

本实验使用**相对容差**而非绝对容差：

```c
if (fabs(ref[i] - test[i]) > rtol * (1.0 + fabs(ref[i]))) return 0;
```

原因是 V2 与 V4/V5 都改变了浮点加法的次序，而误差的绝对量级随 $n$ 增长。$n = 10^6$ 时每行的和约为 $5 \times 10^5$，其舍入误差已远超 $10^{-8}$；若沿用固定的绝对容差，就得随问题规模反复调整。这与实验二 11 节 ⑤ 的结论一致：**容差必须与算法的数值特性和问题规模相匹配**。


## 8. 源代码、编译与运行


In [ ]:
%%writefile {SRC_DIR}/omp_gemv.c
#define _POSIX_C_SOURCE 200809L

#include <math.h>
#include <omp.h>
#include <stdio.h>
#include <stdlib.h>
#include <time.h>

#ifndef _OPENMP
#error "OpenMP is required. Please compile with -fopenmp."
#endif

#define NTIMES 5
#define MAX_THREADS 64
#define ALIGN_BYTES 64
#define RTOL 1e-10

#define BANNER "============================================================"
#define LINE "------------------------------------------------------------"

// ----------------------------------------------------------------------------
// Common helpers
// ----------------------------------------------------------------------------
static double get_time_ms(void) {
  struct timespec ts;
  clock_gettime(CLOCK_MONOTONIC, &ts);
  return (double)ts.tv_sec * 1000.0 + (double)ts.tv_nsec / 1000000.0;
}

static void *alloc_aligned(size_t bytes) {
  size_t rounded = ((bytes + ALIGN_BYTES - 1) / ALIGN_BYTES) * ALIGN_BYTES;
  return aligned_alloc(ALIGN_BYTES, rounded);
}

// Relative check. The column and block partitions reorder the additions, so an
// absolute tolerance would have to be retuned for every problem size.
static int check_rel(const double *ref, const double *test, long n,
                     double rtol) {
  for (long i = 0; i < n; i++) {
    if (fabs(ref[i] - test[i]) > rtol * (1.0 + fabs(ref[i]))) {
      return 0;
    }
  }
  return 1;
}

static void print_table_header(void) {
  printf("\n%s\n", LINE);
  printf("| %-26s | %9s | %7s | %-5s |\n", "Method", "Time(ms)", "Speedup",
         "Check");
  printf("|----------------------------|-----------|---------|-------|\n");
}

static void print_row(const char *name, double time_ms, double base_ms,
                      int check) {
  const char *status = (check < 0) ? "-" : (check ? "PASS" : "FAIL");
  double speedup = (time_ms > 0.0) ? base_ms / time_ms : 0.0;
  printf("| %-26s | %9.3f | %6.2fx | %-5s |\n", name, time_ms, speedup, status);
}

// ============================================================================
// V0: serial baseline
// ============================================================================
static void gemv_serial(const double *A, const double *x, double *y, long m,
                        long n) {
  for (long i = 0; i < m; i++) {
    double sum = 0.0;
    const double *row = A + i * n;
    for (long j = 0; j < n; j++) {
      sum += row[j] * x[j];
    }
    y[i] = sum;
  }
}

// ============================================================================
// V1: row partition. Each thread owns whole rows, so the writes to y never
// overlap and no reduction is required. At most min(m, p) threads are used.
// ============================================================================
static void gemv_row(const double *A, const double *x, double *y, long m,
                     long n, int thread_count) {
#pragma omp parallel for num_threads(thread_count) default(none) \
    shared(A, x, y, m, n)
  for (long i = 0; i < m; i++) {
    // Declared inside the loop body, therefore private by construction.
    double sum = 0.0;
    const double *row = A + i * n;
    for (long j = 0; j < n; j++) {
      sum += row[j] * x[j];
    }
    y[i] = sum;
  }
}

// ============================================================================
// V2: column partition. Every thread works on a slice of one row at a time.
// This creates m parallel regions, m reductions and m implicit barriers, so its
// cost is proportional to m and to nothing else.
// ============================================================================
static void gemv_col(const double *A, const double *x, double *y, long m,
                     long n, int thread_count) {
  for (long i = 0; i < m; i++) {
    double sum = 0.0;
    const double *row = A + i * n;

#pragma omp parallel for num_threads(thread_count) \
    reduction(+ : sum) default(none) shared(row, x, n)
    for (long j = 0; j < n; j++) {
      sum += row[j] * x[j];
    }
    y[i] = sum;
  }
}

// ============================================================================
// V3: row partition plus SIMD. Thread level parallelism on the outer loop and
// data level parallelism on the inner one. The two layers are orthogonal.
// ============================================================================
static void gemv_row_simd(const double *A, const double *x, double *y, long m,
                          long n, int thread_count) {
#pragma omp parallel for num_threads(thread_count) default(none) \
    shared(A, x, y, m, n)
  for (long i = 0; i < m; i++) {
    double sum = 0.0;
    const double *row = A + i * n;

#pragma omp simd reduction(+ : sum)
    for (long j = 0; j < n; j++) {
      sum += row[j] * x[j];
    }
    y[i] = sum;
  }
}

// ============================================================================
// V4: 2D partition. Every row is split into nb column blocks, and the block
// sums are stored in partial[]. The (i, b) nest is perfectly nested and fully
// independent, so collapse(2) can distribute all m * nb points.
// ============================================================================
static void gemv_block(const double *A, const double *x, double *y,
                       double *partial, long m, long n, long nb,
                       int thread_count) {
#pragma omp parallel for collapse(2) num_threads(thread_count) default(none) \
    shared(A, x, partial, m, n, nb)
  for (long i = 0; i < m; i++) {
    for (long b = 0; b < nb; b++) {
      double s = 0.0;
      const double *row = A + i * n;
      // Bounds derived from b alone, so n need not be divisible by nb.
      const long lo = b * n / nb;
      const long hi = (b + 1) * n / nb;
      for (long j = lo; j < hi; j++) {
        s += row[j] * x[j];
      }
      partial[i * nb + b] = s;
    }
  }

  for (long i = 0; i < m; i++) {
    double sum = 0.0;
    for (long b = 0; b < nb; b++) {
      sum += partial[i * nb + b];
    }
    y[i] = sum;
  }
}

// ============================================================================
// V5: 2D partition plus SIMD. collapse(2) merges the two outer levels, and the
// innermost block loop is handed to the vector unit.
// ============================================================================
static void gemv_block_simd(const double *A, const double *x, double *y,
                            double *partial, long m, long n, long nb,
                            int thread_count) {
#pragma omp parallel for collapse(2) num_threads(thread_count) default(none) \
    shared(A, x, partial, m, n, nb)
  for (long i = 0; i < m; i++) {
    for (long b = 0; b < nb; b++) {
      double s = 0.0;
      const double *row = A + i * n;
      const long lo = b * n / nb;
      const long hi = (b + 1) * n / nb;

#pragma omp simd reduction(+ : s)
      for (long j = lo; j < hi; j++) {
        s += row[j] * x[j];
      }
      partial[i * nb + b] = s;
    }
  }

  for (long i = 0; i < m; i++) {
    double sum = 0.0;
    for (long b = 0; b < nb; b++) {
      sum += partial[i * nb + b];
    }
    y[i] = sum;
  }
}

int main(int argc, char *argv[]) {
  if (argc != 4) {
    printf("Usage: %s <m> <n> <thread_count>\n", argv[0]);
    printf("Example (square): %s 4000 4000 16\n", argv[0]);
    printf("Example (flat)  : %s 4 1000000 16\n", argv[0]);
    return 1;
  }

  long m = strtol(argv[1], NULL, 10);
  long n = strtol(argv[2], NULL, 10);
  int thread_count = (int)strtol(argv[3], NULL, 10);

  if (m <= 0 || n <= 0) {
    printf("Error: m and n must be > 0\n");
    return 1;
  }
  if (thread_count < 1 || thread_count > MAX_THREADS) {
    printf("Error: thread_count must be between 1 and %d\n", MAX_THREADS);
    return 1;
  }

  // One column block per thread is enough to fill the team for any m >= 1.
  long nb = thread_count;
  if (nb > n) {
    nb = n;
  }

  size_t matrix_bytes = (size_t)m * (size_t)n * sizeof(double);

  printf("%s\n", BANNER);
  printf(" Lab 7: Matrix-Vector Multiplication (y = A * x)\n");
  printf(" Matrix: %ld x %ld (%.1f MB) | Threads: %d | Runs: %d\n", m, n,
         (double)matrix_bytes / (1024.0 * 1024.0), thread_count, NTIMES);
  printf(" Column blocks nb: %ld | Collapsed iterations: %ld\n", nb, m * nb);
  printf(" _OPENMP: %d | Procs: %d\n", _OPENMP, omp_get_num_procs());
  printf("%s\n", BANNER);

  if (m >= thread_count) {
    printf("\n[Shape] m >= thread_count: the row partition alone can already\n");
    printf("        fill the team, so collapse(2) is not expected to help.\n");
  } else {
    printf("\n[Shape] m < thread_count: the row partition can use at most %ld\n",
           m);
    printf("        of the %d threads, so collapse(2) should help here.\n",
           thread_count);
  }

  double *A = (double *)alloc_aligned(matrix_bytes);
  double *x = (double *)alloc_aligned((size_t)n * sizeof(double));
  double *y_ref = (double *)alloc_aligned((size_t)m * sizeof(double));
  double *y_test = (double *)alloc_aligned((size_t)m * sizeof(double));
  double *partial =
      (double *)alloc_aligned((size_t)m * (size_t)nb * sizeof(double));

  if (A == NULL || x == NULL || y_ref == NULL || y_test == NULL ||
      partial == NULL) {
    printf("Error: memory allocation failed (%.1f MB requested)\n",
           (double)matrix_bytes / (1024.0 * 1024.0));
    return 1;
  }

  for (long i = 0; i < m; i++) {
    for (long j = 0; j < n; j++) {
      A[i * n + j] = (double)((i + j) % 100) / 100.0;
    }
  }
  for (long j = 0; j < n; j++) {
    x[j] = 1.0;
  }

  double start = 0.0;
  double t[6] = {0.0};
  int ok[6] = {0};

  for (int r = 0; r < NTIMES; r++) {
    start = get_time_ms();
    gemv_serial(A, x, y_ref, m, n);
    t[0] += get_time_ms() - start;
  }
  ok[0] = -1;

  for (int r = 0; r < NTIMES; r++) {
    start = get_time_ms();
    gemv_row(A, x, y_test, m, n, thread_count);
    t[1] += get_time_ms() - start;
  }
  ok[1] = check_rel(y_ref, y_test, m, RTOL);

  for (int r = 0; r < NTIMES; r++) {
    start = get_time_ms();
    gemv_col(A, x, y_test, m, n, thread_count);
    t[2] += get_time_ms() - start;
  }
  ok[2] = check_rel(y_ref, y_test, m, RTOL);

  for (int r = 0; r < NTIMES; r++) {
    start = get_time_ms();
    gemv_row_simd(A, x, y_test, m, n, thread_count);
    t[3] += get_time_ms() - start;
  }
  ok[3] = check_rel(y_ref, y_test, m, RTOL);

  for (int r = 0; r < NTIMES; r++) {
    start = get_time_ms();
    gemv_block(A, x, y_test, partial, m, n, nb, thread_count);
    t[4] += get_time_ms() - start;
  }
  ok[4] = check_rel(y_ref, y_test, m, RTOL);

  for (int r = 0; r < NTIMES; r++) {
    start = get_time_ms();
    gemv_block_simd(A, x, y_test, partial, m, n, nb, thread_count);
    t[5] += get_time_ms() - start;
  }
  ok[5] = check_rel(y_ref, y_test, m, RTOL);

  for (int i = 0; i < 6; i++) {
    t[i] /= NTIMES;
  }

  print_table_header();
  print_row("V0: Serial Baseline", t[0], t[0], ok[0]);
  print_row("V1: Row Partition", t[1], t[0], ok[1]);
  print_row("V2: Column Partition", t[2], t[0], ok[2]);
  print_row("V3: Row + SIMD", t[3], t[0], ok[3]);
  print_row("V4: 2D + collapse(2)", t[4], t[0], ok[4]);
  print_row("V5: 2D + collapse + SIMD", t[5], t[0], ok[5]);
  printf("%s\n", LINE);

  double mflop = 2.0 * (double)m * (double)n / 1.0e6;
  double best = t[1];
  for (int i = 2; i < 6; i++) {
    if (t[i] < best) {
      best = t[i];
    }
  }
  printf("\nEffective throughput: serial %.3f GFLOPS | best parallel %.3f GFLOPS\n",
         mflop / t[0], mflop / best);
  printf("Effective bandwidth : serial %.2f GB/s   | best parallel %.2f GB/s\n",
         8.0 * (double)m * (double)n / 1.0e6 / t[0],
         8.0 * (double)m * (double)n / 1.0e6 / best);

  free(A);
  free(x);
  free(y_ref);
  free(y_test);
  free(partial);
  return 0;
}


### 8.1 编译


In [ ]:
bin_gemv = compile_c('omp_gemv.c')


### 8.2 运行一 · 方阵（$m \gg p$）

取 $m = n = 4000$、16 线程，矩阵占用约 122 MB。此时外层有 4000 次迭代，远多于线程数，按行划分本身已足以用满线程组。

> **在异构多核平台上，测速前请先设置 `OMP_PROC_BIND=close` 与 `OMP_PLACES=cores`**，否则线程落点的随机性会掩盖本实验要观察的规律。


In [ ]:
M_SQ = N_SQ = 4000          # 方阵规模；内存较小的平台可改为 2000
NT = 16                     # 线程数，按 16 核平台设定

RUN_ENV = {'OMP_PROC_BIND': 'close', 'OMP_PLACES': 'cores'}

out_sq = run_c(bin_gemv, M_SQ, N_SQ, NT, env=RUN_ENV)
rows_sq = parse_table(out_sq)
print()
for name, ms, sp, chk in rows_sq:
    print('  %-28s %9.3f ms  %5.2fx  %s' % (name, ms, sp, chk))


### 8.3 Roofline 校核

把实测的有效算力与 1.4 节的理论分析对照，判断该内核是否确实受带宽限制。


In [ ]:
flops = 2.0 * M_SQ * N_SQ               # 每个元素一次乘、一次加
bytes_read = 8.0 * M_SQ * N_SQ          # 矩阵 A，双精度

print('算术强度 I = %.3f flop/byte' % (flops / bytes_read))
print()
print('%-28s %10s %13s %15s' % ('版本', '耗时(ms)', '算力(GFLOPS)',
                                '等效带宽(GB/s)'))
for name, ms, sp, chk in rows_sq:
    print('%-28s %10.3f %13.3f %15.2f'
          % (name, ms, flops / 1e6 / ms, bytes_read / 1e6 / ms))
print()
print('若 V1、V3、V4、V5 的等效带宽都停在某个相近的数值附近不再提升，')
print('则该数值即为本机的有效内存带宽，内核确实受带宽限制。')


### 8.4 运行二 · 扁矩阵（$m < p$）

保持内核不变，只改变矩阵的形状：取 $m = 4$、$n = 1000000$，仍用 16 线程。矩阵元素总数为 $4 \times 10^6$，占用约 30 MB。

此时按行划分**最多只能用上 4 个线程**，其余十二个线程分不到任何迭代；而二维划分的迭代空间有 $4 \times 16 = 64$ 个点，足以分给全部线程。

程序会根据 $m$ 与线程数的关系自动打印形状提示。


In [ ]:
M_FLAT, N_FLAT = 4, 1000000     # 扁矩阵：行数 4 < 线程数 16

out_flat = run_c(bin_gemv, M_FLAT, N_FLAT, NT, env=RUN_ENV)
rows_flat = parse_table(out_flat)
print()
for name, ms, sp, chk in rows_flat:
    print('  %-28s %9.3f ms  %5.2fx  %s' % (name, ms, sp, chk))


### 8.5 对照 · 中等形状（$m > p$）

把行数增大到 64（大于线程数 16），保持元素总数不变（$n$ 相应减小为 62500）。此时按行划分已能用满全部线程，`collapse` 的收益应当消失。

三组运行的计算量关系：运行二与运行三完全相同（均为 $4 \times 10^6$ 个元素），运行一是它们的四倍。


In [ ]:
M_MID, N_MID = 64, 62500        # 中等形状：行数 64 > 线程数 16，元素总数与扁矩阵相同

out_mid = run_c(bin_gemv, M_MID, N_MID, NT, env=RUN_ENV)
rows_mid = parse_table(out_mid)

import unicodedata


def disp_width(s):
    """按显示宽度计算字符串长度：中日韩全角字符计 2，其余计 1。"""
    return sum(2 if unicodedata.east_asian_width(ch) in 'WF' else 1 for ch in s)


def pad(s, width, right=True):
    """按显示宽度补齐，使中英混排的表格能够对齐。"""
    fill = ' ' * max(0, width - disp_width(s))
    return fill + s if right else s + fill


LABELS = ('V1: Row Partition', 'V2: Column Partition', 'V3: Row + SIMD',
          'V4: 2D + collapse(2)', 'V5: 2D + collapse + SIMD')
HEADS = ('V1 按行', 'V2 按列', 'V3 行+SIMD', 'V4 二维', 'V5 二维+SIMD')
COLW = 14

print()
print('=' * 92)
print(' 三组形状汇总（加速比，基准为各自的串行版本）')
print('=' * 92)
print(pad('形状', 26, right=False) + ''.join(pad(h, COLW) for h in HEADS))
for label, rows in (('方阵   %d x %d' % (M_SQ, N_SQ), rows_sq),
                    ('扁矩阵 %d x %d' % (M_FLAT, N_FLAT), rows_flat),
                    ('中等   %d x %d' % (M_MID, N_MID), rows_mid)):
    d = {r[0]: r[2] for r in rows}
    cells = ''.join(pad('%.2fx' % d.get(k, 0.0), COLW) for k in LABELS)
    print(pad(label, 26, right=False) + cells)
print()
print('预期：V2 的表现随 m 增大而急剧恶化；V4、V5 相对 V1、V3 的优势')
print('      只在第二行（m < 线程数）出现。')


### 8.6 三组结果可视化


In [ ]:
plot_speedup(rows_sq,
             'Lab 7: Square %d x %d, %d threads (m >> p)' % (M_SQ, N_SQ, NT))
plot_speedup(rows_flat,
             'Lab 7: Flat %d x %d, %d threads (m < p)' % (M_FLAT, N_FLAT, NT),
             figsize=(10, 4.5))
plot_speedup(rows_mid,
             'Lab 7: Medium %d x %d, %d threads (m > p)' % (M_MID, N_MID, NT),
             figsize=(10, 4.5))


## 9. 结果分析

> 以下结论针对**趋势规律**。具体数值随平台、内存带宽与向量宽度而变化。

### 9.1 运行一 · 方阵（$m \gg p$）

**① V1 的加速比明显低于线程数**

这与 1.4 节的 Roofline 分析一致。该内核的算术强度只有 0.25 flop/byte，属于访存受限。增加核心数不会增加内存带宽，因而加速比会在带宽饱和处停止增长。

8.3 节的等效带宽表可以直接验证这一点：若各并行版本的等效带宽都停在某个相近的数值上，那个数值就是本机的有效内存带宽。

**② V2 严重慢于串行**

$m = 4000$ 次并行区域创建，按实验五测得的约 $9\,\mu s$ 计，仅此一项开销即约合 36 毫秒，而整个串行计算的耗时不过十余毫秒。归约与栅栏同样各执行 4000 次。

**③ V3 相对 V1 有稳定但有限的提升**

这一提升**主要**来自向量化：V1 与 V3 的线程数、划分方式、访存模式完全相同，唯一的差别是 `#pragma omp simd` 授权了运算重排（`simd` 还可能影响循环展开与指令调度，但相对次要）。

提升幅度有限的原因仍然是带宽：SIMD 加快的是计算，而计算本非瓶颈。理论上双精度 NEON 可提供 2 倍的计算吞吐，但实测收益通常远小于此。

> **一个值得注意的现象**：即便在**单线程**条件下，V3 相对 V1 依然有明显提升（本平台实测约 1.5 倍，具体数值随处理器与编译器而异）。这表明该提升主要来自向量化而非多线程，同时也支持「在不使用 `-ffast-math` 时，`-O3` 不会自动向量化浮点归约循环」这一结论——否则 V1 应已被向量化，二者不应存在明显差异。

**④ V4 与 V1 接近，V5 与 V3 接近**

这一条是运行二的对照基准。方阵上按行划分本已用满线程，`collapse(2)` 无从发挥，因此 V4 与 V1、V5 与 V3 的差异应当很小。**这说明二维划分这一代码重构本身是性能中性的**——有了这个结论，运行二中 V4 相对 V1 的差距才能干净地归因于 `collapse`，而不是归因于重构。

**⑤ 校验全部通过，说明改变求和次序未破坏数值正确性**

V2 与 V4/V5 都改变了浮点加法的次序，结果与串行版本存在末位差异，但均落在相对容差之内。这说明：**允许运算重排并不意味着结果不可接受**，关键在于所选取的容差是否与算法的数值特性相匹配。

### 9.2 运行二 · 扁矩阵（$m < p$）

**⑥ V1 与 V3 被行数卡住**

$m = 4$、16 线程时，按行划分最多只能用上 4 个线程，线程利用率仅 25%。此时增加线程数不会带来任何改善——瓶颈不在带宽，而在**可分配的迭代数量**。

**⑦ V4 与 V5 用满线程组**

二维划分把迭代空间扩大到 $4 \times 16 = 64$ 个点，全部线程都能分到工作。结合 ④ 的结论，V4 相对 V1 的提升即为 `collapse(2)` 的净收益。

**⑧ V2 在此不再是反模式**

这是本实验最值得琢磨的一处观察。$m = 4$ 时，V2 只创建 4 次并行区域（约几十微秒），相对于毫秒级的计算完全可以忽略；而它对 $n = 10^6$ 的内层做归约，恰好能用满全部线程。同一段代码，在方阵上是灾难，在扁矩阵上却是个合理方案。

由此可以把「并行内层是反模式」这条规则修正成它本该有的形态：

> 判据不是「内层还是外层」，而是**并行区域的创建次数与每次区域内有效工作量之比**。这与实验五的粒度原则是同一条原则。

V2 与 V4 在扁矩阵上的差别因此变得很小，两者的取舍转为：V2 需要 $m$ 次归约但代码更短，V4 只需一次合并但要额外的 `partial` 数组。

### 9.3 对照 · 中等形状（$m > p$）

**⑨ `collapse(2)` 的收益消失**

$m = 64 > 16$ 时，按行划分已能用满全部线程，此时 `collapse` 不仅无益，还可能因索引计算开销增加而略慢。V2 的表现则介于两次运行之间——64 次并行区域，代价已开始显现但尚未致命，正好印证了 ⑧ 中「代价正比于 $m$」的判断。

### 9.4 综合：方案选择取决于矩阵形状

把三组结果并列，可以得到一张直接可用的决策表：

| 矩阵形状 | 推荐版本 | 理由 |
|---|---|---|
| $m \gg p$（方阵、高矩阵） | **V3**　按行划分 + SIMD | 按行划分已用满线程且无需归约；二维划分徒增复杂度 |
| $m \approx p$ 或略大 | **V3** | 同上；`collapse` 的收益已不足以抵消其开销 |
| $m < p$（扁矩阵） | **V5**　二维划分 + `collapse` + SIMD | 按行划分用不满线程，需要更细的迭代空间 |
| $m$ 极小（如 1～2） | **V5** 或 V2 | 二者接近；V2 代码更短，V5 少了 $m$ 次 Fork-Join |

**⑩ 但 `collapse` 提高的是线程利用率，不是带宽**

需要注意：无论形状如何变化，本内核的算术强度始终是 0.25 flop/byte。`collapse` 解决的是**线程利用率**问题，而不是带宽问题。因此在扁矩阵上，V5 相对 V1 的提升也会在带宽饱和处封顶，不会达到 $p / \min(m, p) = 4$ 倍的理论值。这与 1.4 节的 Roofline 分析完全一致——三组运行在此收束于同一条结论：

> **先判断瓶颈是什么，再选择手段。**线程利用率不足就改划分方式，带宽不足则只能改算法或数据布局。


## 10. 🔧 动手练习

**练习 1**　把线程数依次取 1、2、4、8、16，在方阵上观察 V1 与 V3 的加速比曲线。该曲线在何处开始趋于平坦？把该点对应的等效带宽记录下来。

**练习 2**　加上 `-ffast-math` 重新编译，观察 V1 是否被自动向量化（V1 与 V3 的差距是否消失）。同时检查校验是否仍然通过。

**练习 3**　把矩阵改为**单精度** `float`。算术强度会变为多少？V3 的向量化收益会如何变化？

**练习 4**　把方阵规模依次取 500×500、1000×1000、2000×2000、4000×4000，观察加速比随规模的变化。小规模时矩阵可完全装入缓存，此时内核是否仍然受带宽限制？

**练习 5**（进阶）　用 `objdump -d` 反汇编 V1 与 V3 的内层循环，确认 V3 中出现了 `fmla v*.2d` 之类的向量指令，而 V1 中只有标量的 `fmadd`。（在 x86-64 平台上，对应的向量指令为 `vfmadd*pd` 或 `mulpd`/`addpd`，标量形式则为 `vfmadd*sd`。）

**练习 6**（综合）　固定矩阵元素总数 $mn = 4 \times 10^6$，令 $m$ 依次取 1、2、4、8、16、32、64、256、1024，逐一运行并记录每种形状下最快的版本。画出「最优版本随 $m$ 迁移」的图，并找出 V1 与 V4 表现相当的**交叉点**。该交叉点与线程数是什么关系？

**练习 7**（进阶）　为 V2 写一个改进版本：把 `#pragma omp parallel` 提到外层循环之外，内层只用 `#pragma omp for reduction`。这样能否挽回 V2 在方阵上的大部分损失？请实测并解释。（提示：回顾实验五 V1 与 V2 的关系。）

**练习 8**（进阶）　V4 中取 $nb = $ 线程数。试着改为 $2p$ 与 $p/2$，在扁矩阵上重新测量。块数变多或变少各会带来什么影响？


### 10.1 练习 1 的参考实现：线程数扩展性与等效带宽


In [ ]:
import matplotlib.pyplot as plt

threads = [1, 2, 4, 8, 16]
bytes_read = 8.0 * M_SQ * N_SQ
sp1, sp3, bw3 = [], [], []
env_full = dict(os.environ)
env_full.update(RUN_ENV)

for nt in threads:
    out = subprocess.run([bin_gemv, str(M_SQ), str(N_SQ), str(nt)],
                         capture_output=True, text=True, env=env_full).stdout
    d = {r[0]: r for r in parse_table(out)}
    sp1.append(d['V1: Row Partition'][2])
    sp3.append(d['V3: Row + SIMD'][2])
    bw3.append(bytes_read / 1e6 / d['V3: Row + SIMD'][1])
    print('线程数 %-2d   V1 %5.2fx   V3 %5.2fx   V3 等效带宽 %6.2f GB/s'
          % (nt, sp1[-1], sp3[-1], bw3[-1]))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))
ax1.plot(threads, sp1, marker='o', label='V1: Row Partition')
ax1.plot(threads, sp3, marker='s', label='V3: Row + SIMD')
ax1.plot(threads, threads, color='gray', linestyle=':', label='Ideal')
ax1.set_xlabel('Thread count')
ax1.set_ylabel('Speedup')
ax1.set_title('Lab 7: Scalability (square matrix)')
ax1.set_xticks(threads)
ax1.grid(linestyle=':', alpha=0.5)
ax1.legend()
ax2.plot(threads, bw3, marker='^', color='#d62728')
ax2.set_xlabel('Thread count')
ax2.set_ylabel('Effective bandwidth [GB/s]')
ax2.set_title('Lab 7: Effective Memory Bandwidth (V3)')
ax2.set_xticks(threads)
ax2.grid(linestyle=':', alpha=0.5)
plt.tight_layout()
plt.show()
print()
print('右图若在某个线程数之后趋于水平，该水平值即为本机有效带宽上限。')


### 10.2 练习 2 的参考实现：`-ffast-math` 的影响


In [ ]:
src_path = os.path.join(SRC_DIR, 'omp_gemv.c')
env_full = dict(os.environ)          # 本单元格可独立运行，不依赖 10.1 节
env_full.update(RUN_ENV)

for flags in (['-O3', '-fopenmp'],
              ['-O3', '-fopenmp', '-ffast-math']):
    tag = 'fast' if '-ffast-math' in flags else 'plain'
    binary = os.path.join(SRC_DIR, 'gemv_' + tag)
    subprocess.run(['gcc'] + flags + ['-o', binary, src_path, '-lm'],
                   check=True)
    out = subprocess.run([binary, '2000', '2000', str(NT)],
                         capture_output=True, text=True,
                         env=env_full).stdout
    d = {r[0]: r for r in parse_table(out)}
    v1, v3 = d['V1: Row Partition'], d['V3: Row + SIMD']
    print('%-12s  V1 %8.3f ms (%s)   V3 %8.3f ms (%s)   V1/V3 = %.2f'
          % (' '.join(flags[2:]) or '(默认)', v1[1], v1[3],
             v3[1], v3[3], v1[1] / v3[1]))
print()
print('若加上 -ffast-math 后 V1/V3 明显趋近 1.00，')
print('说明 V1 的内层循环也被自动向量化了，两者不再有差别。')


## 11. 🤔 思考题

**思考题 1**　V1 中 `y[i]` 由不同线程写入。在什么条件下这些写入会引发实验六讨论的伪共享？本实验的三组参数是否满足该条件？

**思考题 2**　`#pragma omp simd` 与 `#pragma omp for simd` 有何区别？后者的语义是什么？在本实验的 V3 中能否用后者替代？

**思考题 3**　1.4 节计算算术强度时，忽略了向量 $x$ 与结果 $y$ 的访存量。请说明这一忽略在什么条件下成立，以及当 $n$ 很大、$x$ 无法常驻缓存时结论会如何变化。

**思考题 4**　`collapse(2)` 要求两层循环完美嵌套。下面这段代码为何不能 collapse？应如何改写？

```c
for (long i = 0; i < m; i++) {
  double row_scale = 1.0 / (i + 1);
  for (long j = 0; j < n; j++) {
    c[i * n + j] = a[i * n + j] * row_scale;
  }
}
```

**思考题 5**　V2 的代价为何正比于 $m$ 而与 $n$ 无关？请据此推导：给定单次 Fork-Join 代价 $t_{fj}$ 与串行总耗时 $T$，$m$ 小到什么程度时 V2 的额外开销可以忽略（例如低于 5%）？

**思考题 6**　实验五指出「粗粒度优于细粒度」，本实验指出「并行外层优于并行内层」。9.2 节 ⑧ 又说明后者在 $m$ 很小时不成立。这三句话是否矛盾？请给出一个能同时涵盖它们的统一判据。

**思考题 7**　V4 取 $nb = p$。若取 $nb = 1$，V4 会退化成哪个版本？若取 $nb = n$ 呢？由此说明 $nb$ 的选择在什么区间内才有意义。

**思考题 8**（综合）　假设某平台有 8 个核心、向量宽度可容纳 4 个双精度数、内存带宽 20 GB/s、峰值浮点算力 100 GFLOPS。对本实验的矩阵向量乘法内核，请用 Roofline 模型预估其可达到的最高算力，并说明为何单纯增加核心数或加宽向量都无法提升该内核的性能。什么样的改动才可能有效？


## 12. 📌 本实验小结

| 概念 | 要点 |
|---|---|
| 并行层次选择 | 优先并行迭代次数多且独立的**最外层**；「次数足够多」是必要前提 |
| 三种划分方式 | 按行（$\min(m,p)$ 线程、无归约）、按列（$p$ 线程、$m$ 次归约与 Fork-Join）、二维（$p$ 线程、一次廉价合并） |
| 并行内层的代价 | $m$ 次并行区域 + $m$ 次归约 + $m$ 次栅栏，**正比于 $m$，与 $n$ 无关** |
| `#pragma omp simd` | 对编译器的**授权**，允许改变运算次序；是否真的向量化仍由编译器决定 |
| 浮点归约与向量化 | `-O3` 不会自动重排浮点加法，需显式授权 |
| `-ffast-math` vs `simd` | 前者全局生效，后者仅限该循环，风险可控 |
| TLP × DLP | 两个层次正交，可叠加；V1→V3 与 V4→V5 各验证一次 |
| `collapse(k)` | 合并迭代空间；需完美嵌套、循环界符合规范 |
| 归约与 `collapse` | `collapse` 只合并迭代空间，不消除归约；需先用部分和数组把内层归约改造为独立迭代 |
| `collapse` 适用判据 | 仅当外层迭代次数与线程数相当或更少时有益 |
| 校验容差 | 改变求和次序后误差随规模增长，应采用相对容差 |
| Roofline | $I = 0.25$ flop/byte，访存受限，加速比由带宽决定 |

### 一条重要的分析习惯

> 在进行优化之前，应先估算内核的**算术强度**。若内核为访存受限，则单纯增加线程数、加宽向量或改进调度往往收效有限；更有效的手段是**改善数据布局或提高数据复用率**。

本实验的矩阵向量乘法中，$A$ 的每个元素只被使用一次，复用率为零，因而在该算法形式下无法摆脱带宽限制。作为对照，矩阵**矩阵**乘法的算术强度可以通过分块提高到与块大小同阶，那才是一个计算受限的问题——这也正是本章大实验将要讨论的内容。

### 按形状选择方案

> | 矩阵形状 | 推荐版本 |
> |---|---|
> | $m \gg p$ | V3　按行划分 + SIMD |
> | $m \approx p$ 或略大 | V3 |
> | $m < p$ | V5　二维划分 + `collapse(2)` + SIMD |
>
> 这张表真正要传达的不是三行结论，而是它背后的方法：**同一个算法在不同形状的输入上，最优的并行组织方式并不相同**。判断瓶颈在哪一层（线程利用率、同步开销、还是访存带宽），比记住某一种写法更重要。

### 与后续实验的衔接

本章至此讨论的全部是**规则**的并行：迭代空间事先已知，工作量可以预先划分。下一个实验面对的是**不规则**并行——递归算法的任务树在运行时才逐步展开，无法用 `#pragma omp for` 表达。这需要一个全新的构造：`task`。
